In [ ]:
!pip install -q transformers datasets scikit-learn

In [ ]:
# Import relevant libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import datasets - LIAR, ISOT & FakeNewsNet
# LIAR dataset
df_liar_train = pd.read_csv("/kaggle/input/datasets/csmalarkodi/liar-fake-news-dataset/train.tsv", sep="\t", header=None)
df_liar_valid = pd.read_csv("/kaggle/input/datasets/csmalarkodi/liar-fake-news-dataset/valid.tsv", sep="\t", header=None)
df_liar_test = pd.read_csv("/kaggle/input/datasets/csmalarkodi/liar-fake-news-dataset/test.tsv", sep="\t", header=None)
# ISOT dataset
df_isot_fake = pd.read_csv("/kaggle/input/datasets/rahulogoel/isot-fake-news-dataset/News_Dataset/Fake.csv")
df_isot_true = pd.read_csv("/kaggle/input/datasets/rahulogoel/isot-fake-news-dataset/News_Dataset/True.csv")
# FakeNewsNet dataset
df_fakenewsnet_buzzfeed_fake = pd.read_csv("/kaggle/input/datasets/mdepak/fakenewsnet/BuzzFeed_fake_news_content.csv")
df_fakenewsnet_buzzfeed_true = pd.read_csv("/kaggle/input/datasets/mdepak/fakenewsnet/BuzzFeed_real_news_content.csv")
df_fakenewsnet_politifact_fake = pd.read_csv("/kaggle/input/datasets/mdepak/fakenewsnet/PolitiFact_fake_news_content.csv")
df_fakenewsnet_politifact_true = pd.read_csv("/kaggle/input/datasets/mdepak/fakenewsnet/PolitiFact_real_news_content.csv")

In [ ]:
# Add column names to the dataset
df_liar_train.columns = ["ID", "label", "statement", "subjects", "speaker", "speaker_job", "state", "party", 'barely true counts', 'false counts', 'half true counts', 'mostly true counts', 'pants on fire counts', "venue"]
df_liar_valid.columns = ["ID", "label", "statement", "subjects", "speaker", "speaker_job", "state", "party", 'barely true counts', 'false counts', 'half true counts', 'mostly true counts', 'pants on fire counts', "venue"]
df_liar_test.columns = ["ID", "label", "statement", "subjects", "speaker", "speaker_job", "state", "party", 'barely true counts', 'false counts', 'half true counts', 'mostly true counts', 'pants on fire counts', "venue"]

# Dataset with column headings
df_liar_train.head()

In [ ]:
# Remove unneeded columns keeping only label, statement columns
df_liar_train.drop(["ID", "subjects", "speaker", "speaker_job", "state", "party", 'barely true counts', 'false counts', 'half true counts', 'mostly true counts', 'pants on fire counts', "venue"], axis=1, inplace=True)
df_liar_valid.drop(["ID", "subjects", "speaker", "speaker_job", "state", "party", 'barely true counts', 'false counts', 'half true counts', 'mostly true counts', 'pants on fire counts', "venue"], axis=1, inplace=True)
df_liar_test.drop(["ID", "subjects", "speaker", "speaker_job", "state", "party", 'barely true counts', 'false counts', 'half true counts', 'mostly true counts', 'pants on fire counts', "venue"], axis=1, inplace=True)

# Combine all 3 sets into one coherent corpus
df_liar_combined = pd.concat([df_liar_train,df_liar_valid, df_liar_test],axis=0).sample(frac = 1, random_state = 42).reset_index(drop = True)

# We'll need our ensemble to produce a real(true) or fake(false) classification. So we'll only keep rows with either true or false labels.
df_liar_filtered = df_liar_combined[df_liar_combined['label'].isin(['true', 'false'])]

# Replace 
df_liar_filtered['label'] = df_liar_filtered['label'].str.lower().map({'true': 1, 'false': 0})

df_liar_filtered['label'].value_counts()

In [ ]:
# Clean null values & duplicate records
df_liar_filtered = df_liar_filtered.drop_duplicates()
df_liar_filtered.reset_index(drop=True, inplace=True)

print("LIAR shape: ", df_liar_filtered.shape)
print("LIAR Duplicates count:", df_liar_filtered.duplicated().sum())
print("LIAR Null Values: \n", df_liar_filtered.isnull().sum())

In [ ]:
# Create labels for the ISOT datasets
df_isot_fake["label"] = 0
df_isot_true["label"] = 1
df_isot_combined = pd.concat([df_isot_fake,df_isot_true],axis=0).sample(frac = 1, random_state = 42).reset_index(drop = True)

# Remove date and subject columns
df_isot_combined.drop(["date", "subject"], axis=1, inplace=True)

# Check for null values  and duplicates
print("ISOT Duplicates count:", df_isot_combined.duplicated().sum())
print("ISOT Null Values: \n", df_isot_combined.isnull().sum())

In [ ]:
# Clean duplicate records
df_isot_filtered = df_isot_combined.drop_duplicates()
df_isot_filtered.reset_index(drop=True, inplace=True)

# Create new 'statement' column
df_isot_filtered = df_isot_filtered.copy()
df_isot_filtered["statement"] = df_isot_filtered["title"] + " " + df_isot_filtered["text"]

print("ISOT Shape: ", df_isot_filtered.shape)
print("ISOT Duplicates count:", df_isot_filtered.duplicated().sum())
print("ISOT Null Values: \n", df_isot_filtered.isnull().sum())

In [ ]:
# remove unneeded columns
df_fakenewsnet_buzzfeed_fake.drop(["id", "url", "top_img", "authors", "source", "publish_date", "movies", "images", "canonical_link", "meta_data"], axis=1, inplace=True)
df_fakenewsnet_buzzfeed_true.drop(["id", "url", "top_img", "authors", "source", "publish_date", "movies", "images", "canonical_link", "meta_data"], axis=1, inplace=True)
df_fakenewsnet_politifact_fake.drop(["id", "url", "top_img", "authors", "source", "publish_date", "movies", "images", "canonical_link", "meta_data"], axis=1, inplace=True)
df_fakenewsnet_politifact_true.drop(["id", "url", "top_img", "authors", "source", "publish_date", "movies", "images", "canonical_link", "meta_data"], axis=1, inplace=True)

# Create new 'label' column
df_fakenewsnet_buzzfeed_fake["label"] = 0
df_fakenewsnet_buzzfeed_true["label"] = 1
df_fakenewsnet_politifact_fake["label"] = 0
df_fakenewsnet_politifact_true["label"] = 1

# Create new 'statement' column
df_fakenewsnet_buzzfeed_fake["statement"] = df_fakenewsnet_buzzfeed_fake["title"] + " " + df_fakenewsnet_buzzfeed_fake["text"]
df_fakenewsnet_buzzfeed_true["statement"] = df_fakenewsnet_buzzfeed_true["title"] + " " + df_fakenewsnet_buzzfeed_true["text"]
df_fakenewsnet_politifact_fake["statement"] = df_fakenewsnet_politifact_fake["title"] + " " + df_fakenewsnet_politifact_fake["text"]
df_fakenewsnet_politifact_true["statement"] = df_fakenewsnet_politifact_true["title"] + " " + df_fakenewsnet_politifact_true["text"]

# Combine the modified datasets
df_fakenewsnet_combined = pd.concat([df_fakenewsnet_buzzfeed_fake,df_fakenewsnet_buzzfeed_true, df_fakenewsnet_politifact_fake, df_fakenewsnet_politifact_true],axis=0).sample(frac = 1, random_state = 42).reset_index(drop = True)

df_fakenewsnet_combined.sample(5)

In [ ]:
# Clean duplicate records
df_fakenewsnet_filtered = df_fakenewsnet_combined.drop_duplicates()
df_fakenewsnet_filtered.reset_index(drop=True, inplace=True)

print("FakeNewsNet Shape: ", df_fakenewsnet_filtered.shape)
print("FakeNewsNet Duplicates count:", df_fakenewsnet_filtered.duplicated().sum())
print("FakeNewsNet Null Values: \n", df_fakenewsnet_filtered.isnull().sum())

In [ ]:
# Faqat article datasetlar
df_article = pd.concat(
    [df_isot_filtered, df_fakenewsnet_filtered],
    axis=0
).sample(frac=1, random_state=42).reset_index(drop=True)

# Faqat kerakli ustunlar
df_article = df_article[["statement", "label"]]

print("Shape:", df_article.shape)
print(df_article["label"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_article,
    test_size=0.2,
    stratify=df_article["label"],
    random_state=42
)

print("Train:", train_df.shape)
print("Test:", test_df.shape)

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

train_dataset = train_dataset.remove_columns(["__index_level_0__"])
test_dataset = test_dataset.remove_columns(["__index_level_0__"])

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["statement"],
        truncation=True,
        padding="max_length",
        max_length=512 
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    load_best_model_at_end=True
)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions)
    }

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)
trainer.train()
trainer.evaluate()

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def predict_article(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)

    pred = torch.argmax(probs, dim=1).item()

    return {
        "prediction": "Real" if pred == 1 else "Fake",
        "confidence": round(probs[0][pred].item() * 100, 2)
    }

In [ ]:
samples = [
"SINGAPORE/DUBAI, March 6 (Reuters)-Soon after ​the first Iranian missile and drone attacks on Dubai last week, two Indian entrepreneurs based there tried to move more than $100,000 each from their local bank ‌accounts to Singapore to hedge risk.Technological glitches in the aftermath of the Iranian attacks initially scuppered those plans, the entrepreneurs, who did not wish to be identified due to the sensitivity of the matter, told Reuters."
]

for text in samples:
    print("="*80)
    print(text)
    print(predict_article(text))

In [ ]:
import shutil
trainer.save_model("./best_model")
tokenizer.save_pretrained("./best_model")
shutil.make_archive("best_model", "zip", "./best_model")

In [ ]:
#LIAR dataset

In [ ]:
import pandas as pd

df_train = pd.read_csv("/kaggle/input/datasets/csmalarkodi/liar-fake-news-dataset/train.tsv", sep="\t", header=None)
df_valid = pd.read_csv("/kaggle/input/datasets/csmalarkodi/liar-fake-news-dataset/valid.tsv", sep="\t", header=None)
df_test = pd.read_csv("/kaggle/input/datasets/csmalarkodi/liar-fake-news-dataset/test.tsv", sep="\t", header=None)

columns = [
    "ID", "label", "statement", "subjects", "speaker", "speaker_job",
    "state", "party", "barely_true_counts", "false_counts",
    "half_true_counts", "mostly_true_counts",
    "pants_fire_counts", "venue"
]

df_train.columns = columns
df_valid.columns = columns
df_test.columns = columns

In [ ]:
df_train = df_train[["statement", "label"]]
df_valid = df_valid[["statement", "label"]]
df_test = df_test[["statement", "label"]]

In [ ]:
real_labels = ["true", "mostly-true", "half-true"]
fake_labels = ["barely-true", "false", "pants-fire"]

def map_label(x):
    if x in real_labels:
        return 1
    else:
        return 0

df_train["label"] = df_train["label"].apply(map_label)
df_valid["label"] = df_valid["label"].apply(map_label)
df_test["label"] = df_test["label"].apply(map_label)

print(df_train["label"].value_counts())

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(df_train)
valid_dataset = Dataset.from_pandas(df_valid)
test_dataset = Dataset.from_pandas(df_test)

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["statement"],
        truncation=True,
        padding="max_length",
        max_length=96
    )

train_dataset = train_dataset.map(tokenize, batched=True)
valid_dataset = valid_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
valid_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./liar_2class_results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=1e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    report_to="none"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics
)
trainer.train()
trainer.evaluate(test_dataset)

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def predict_claim(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=96
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)

    pred = torch.argmax(probs, dim=1).item()

    return {
        "prediction": "Real" if pred == 1 else "Fake",
        "confidence": round(probs[0][pred].item() * 100, 2)
    }

In [ ]:
predict_claim("The economy has grown faster than at any point in the last decade.")

In [ ]:
import shutil
trainer.save_model("./best_liar_model")
tokenizer.save_pretrained("./best_liar_model")
shutil.make_archive("best_liar_model", "zip", "./best_liar_model")

In [ ]:
#Hybrid model

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===== ARTICLE MODEL =====
article_tokenizer = BertTokenizer.from_pretrained("/kaggle/input/models/komronraximov/fake-news-detection-model/pytorch/default/1/best_model")
article_model = BertForSequenceClassification.from_pretrained("/kaggle/input/models/komronraximov/fake-news-detection-model/pytorch/default/1/best_model")
article_model.to(device)
article_model.eval()

# ===== LIAR MODEL =====
liar_tokenizer = BertTokenizer.from_pretrained("/kaggle/input/models/komronraximov/fake-news-detection-model/pytorch/default/1/best_liar_model")
liar_model = BertForSequenceClassification.from_pretrained("/kaggle/input/models/komronraximov/fake-news-detection-model/pytorch/default/1/best_liar_model")
liar_model.to(device)
liar_model.eval()

In [ ]:
def predict_article(text):
    inputs = article_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = article_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)

    pred = torch.argmax(probs, dim=1).item()

    return {
        "prediction": "Real" if pred == 1 else "Fake",
        "confidence": round(probs[0][pred].item() * 100, 2)
    }

In [ ]:
def predict_claim(text):
    inputs = liar_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=96
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = liar_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)

    pred = torch.argmax(probs, dim=1).item()

    return {
        "prediction": "Real" if pred == 1 else "Fake",
        "confidence": round(probs[0][pred].item() * 100, 2)
    }

In [ ]:
def interpret_confidence(score):
    if score >= 85:
        return "Very High"
    elif score >= 70:
        return "High"
    elif score >= 55:
        return "Moderate"
    else:
        return "Low"


def hybrid_predict(text):

    word_count = len(text.split())

    if word_count > 25:
        result = predict_article(text)
        model_used = "Article Model"
    else:
        result = predict_claim(text)
        model_used = "LIAR Claim Model"

    return {
        "prediction": result["prediction"],
        "confidence_percent": result["confidence"],
        "confidence_level": interpret_confidence(result["confidence"]),
        "model_used": model_used,
        "word_count": word_count
    }

In [ ]:
print(hybrid_predict("The election results were secretly manipulated through hidden digital systems."))

In [ ]:
import os
import requests

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

def google_fact_check(query):

    url = "https://factchecktools.googleapis.com/v1alpha1/claims:search"

    params = {
        "query": query,
        "key": GOOGLE_API_KEY,
        "languageCode": "en"
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        return None

    data = response.json()

    if "claims" not in data:
        return None

    claim = data["claims"][0]
    review = claim["claimReview"][0]

    rating = review.get("textualRating", "").lower()
    publisher = review["publisher"]["name"]

    return {
        "rating": rating,
        "publisher": publisher
    }

In [ ]:
def convert_rating_to_score(rating):

    rating = rating.lower()

    if "false" in rating:
        return 1.0  # fake
    elif "mostly false" in rating:
        return 0.8
    elif "half" in rating:
        return 0.5
    elif "mostly true" in rating:
        return 0.2
    elif "true" in rating:
        return 0.0  # real
    else:
        return None

In [ ]:
def advanced_hybrid_predict(text):

    # 1️⃣ Model prediction
    base_result = hybrid_predict(text)
    base_score = base_result["confidence_percent"] / 100

    # 2️⃣ Google fact check
    fact_data = google_fact_check(text)

    if fact_data:

        rating_score = convert_rating_to_score(fact_data["rating"])

        if rating_score is not None:
            final_score = (base_score + rating_score) / 2

            final_label = "Fake" if final_score > 0.6 else "Real"

            return {
                "prediction": final_label,
                "confidence_percent": round(final_score * 100, 2),
                "base_model": base_result,
                "fact_check_rating": fact_data["rating"],
                "fact_check_source": fact_data["publisher"],
                "verification": "Verified by fact-check organization"
            }

    return {
        **base_result,
        "verification": "No fact-check evidence found"
    }

In [ ]:
print(advanced_hybrid_predict(
    "March 6 (Reuters) - A spokesperson for Iran’s Revolutionary Guards challenged ​U.S. President Donald Trump to deploy U.S. ‌naval vessels to escort oil tankers through the Strait of Hormuz, Iranian state media reported on ​Friday.The U.S. Navy could begin escorting oil ​tankers through the Strait of Hormuz ⁠if necessary, Trump said on Tuesday. The conflict ​in the Middle East has halted shipping and ​energy exports through the vital Strait of Hormuz."
))